# 10 · Exploratory: liking, emotion, and the model's internal state

Extensions beyond the Jin replication. **These do not gate the email or the `scene_null.py` ask.**

Three questions from Rhea:
1. Can we compare sentiment scores ↔ model internal state ↔ fMRI ↔ behavior together?  → **§3 variance partitioning**
2. Is there an *inverse* relation between liking a character and learning about them?  → **§1**
3. Does emotion (amygdala) interrupt character understanding?  → **§4**

> [!warning] Build on what works
> Use the **IS-RSA / group-brain** frameworks (validated in 04/05/07). Do **NOT** build on the `06` pattern-shift — its positive control failed (1 ROI, wrong network), so anything layered on it inherits an uninterpretable null.

### Conceptual note — is 'sensory' alignment actually 'feeling'? (§2)
The 768-D embeddings (`04b.1`) recovered ROIs 9 (LH auditory), 60 (RH S2), 78 (temporal pole) — sensory/affective, not the mentalizing network. Rhea's point: *isn't sensory structure closer to 'feeling' — so is that a success, not a failure?*

The catch for an **ML model**: RoBERTa has no senses; it reads text. So an alignment to auditory/somatosensory cortex can't mean the model 'feels' — it means the model's **affective-language geometry** (valence-laden vocabulary, intensity) happens to covary with whatever those ROIs track while subjects watch the actual movie (voices, prosody, depicted touch). That is *semantic evaluation*, which is neither social inference (mentalizing) nor embodied feeling. §2 tests whether the (weak) alignment even favors sensory cortex — and it doesn't, significantly.


## §1 · Does liking a character track how much you *learn* about them?

**'Learning' operationalized 3 ways** (per participant × character): impression **updating** = cumulative |Δ(pos−neg)| across runs; **valence range**; **elaboration** = mean word count of reflections. **'Liking'** = terminal post-scan `t_like`.

Hypotheses: *halo/premature closure* → like↑ ⇒ update↓ (negative). *engagement* → like↑ ⇒ update↑ (positive).

> [!note] Result (run 2026-07-18)
> **Between participant×character:** like ~ updating **rho = +0.234, p = .009**; like ~ valence-range +0.286, p=.001; like ~ elaboration **−0.166, p=.066** (write *less* about liked characters, trend). Per character: Jack +0.43\*, Randall +0.40\*, Kate ~0, Kevin +0.18.
> **Within participant** (across their 4 characters, controls for rating scale): mean rho **+0.085, p=.46 (null)**.
>
> **Read:** the *inverse* hypothesis is **not** supported — if anything it's the opposite: people update more about characters they like (an engagement effect), driven by between-person differences; within a person it's flat. The one hint of an inverse effect is elaboration (fewer words about liked characters), but it's only a trend.


In [6]:
import pandas as pd, numpy as np, re
from scipy.stats import spearmanr, wilcoxon
sc  = pd.read_csv('results/scored/00__reflection_sentiment.csv')
con = pd.read_csv('results/step2/02__final_impression_constructs.csv')
digits = lambda s: re.sub(r'\D','',str(s))
sc['pid']=sc.Participant.map(digits);  sc['Character']=sc.Character.str.lower().str.strip()
con['pid']=con.Participant.map(digits); con['Character']=con.Character.str.lower().str.strip()
sc['val']=sc['Twitter_RoB_pos']-sc['Twitter_RoB_neg']            # sentiment valence per run
rows=[]
for (pid,ch),g in sc.groupby(['pid','Character']):
    g=g.sort_values('Run'); v=g['val'].values; wc=g['WordCount'].values
    rows.append({'pid':pid,'Character':ch,
                 'updating':np.nansum(np.abs(np.diff(v))),                       # cumulative impression movement
                 'valence_range':(np.nanmax(v)-np.nanmin(v)) if len(v)>1 else np.nan,
                 'elaboration':np.nanmean(wc)})
beh=pd.DataFrame(rows).merge(con[['pid','Character','t_like']],on=['pid','Character'],how='inner')
print('units (participant x character):',len(beh))
def rep(x,y,lab):
    d=beh[[x,y]].dropna(); r,p=spearmanr(d[x],d[y]); print(f'  {lab:34s} rho={r:+.3f} p={p:.3f} n={len(d)}')
print('POOLED (between participant x character):')
rep('t_like','updating','like ~ impression updating'); rep('t_like','valence_range','like ~ valence range'); rep('t_like','elaboration','like ~ elaboration')
print('PER CHARACTER (like ~ updating):')
for ch in ['jack','kate','randall','kevin']:
    s=beh[beh.Character==ch][['t_like','updating']].dropna()
    if len(s)>3: r,p=spearmanr(s.t_like,s.updating); print(f'  {ch:9s} rho={r:+.3f} p={p:.3f} n={len(s)}')
ws=[spearmanr(g.t_like,g.updating)[0] for _,g in beh.groupby('pid') if g.t_like.nunique()>1 and len(g)>=3]
ws=np.array([x for x in ws if not np.isnan(x)])
print(f'WITHIN-SUBJECT mean rho(like,updating)={ws.mean():+.3f}  Wilcoxon p={wilcoxon(ws).pvalue:.3f}  (neg => update liked chars less)')


units (participant x character): 124
POOLED (between participant x character):
  like ~ impression updating         rho=+0.234 p=0.009 n=124
  like ~ valence range               rho=+0.286 p=0.001 n=124
  like ~ elaboration                 rho=-0.166 p=0.066 n=124
PER CHARACTER (like ~ updating):
  jack      rho=+0.425 p=0.017 n=31
  kate      rho=-0.066 p=0.723 n=31
  randall   rho=+0.397 p=0.027 n=31
  kevin     rho=+0.180 p=0.333 n=31
WITHIN-SUBJECT mean rho(like,updating)=+0.085  Wilcoxon p=0.461  (neg => update liked chars less)


## §2 · Is the model's alignment 'sensory/feeling' — or just weak everywhere?

Uses the existing layer-wise brain–model RSA (`03b__layerwise_alignment.npy`, 13 layers × 116 ROIs). For each ROI take the best |alignment| across layers, then compare **sensory/motor** ROIs (Somatomotor/Visual/Aud) vs **social/affective** ROIs (DMN/Temporoparietal/Limbic).

> [!note] Result (run 2026-07-18)
> sensory/motor mean peak |r| = **0.033** (max 0.068) vs social/affective **0.028** (max 0.058). Sensory > social **not significant (Mann-Whitney p = 0.120)**. Top-aligned ROIs are a *mix*: 60 S2, 14 SPL, **98 TPJ**, 2 visual, 9 auditory, **97 TPJ**, 78 temporal pole.
>
> **Read:** there is **no meaningful alignment anywhere (all |r| < 0.07)**, and the sensory lean isn't significant — TPJ sits right among the top ROIs. So we can't claim 'the model captures feeling via sensory cortex,' and we also shouldn't claim a clean 'social-network failure.' The honest statement: the model's representational geometry has **no brain-aligned social structure at any depth** — consistent with *semantic evaluation*, not social stance or embodied feeling.


In [7]:
import numpy as np
from roi_labels import ROI_NAME
from scipy.stats import mannwhitneyu
A=np.asarray(np.load('results/jinrep/03b__layerwise_alignment.npy',allow_pickle=True).item()['align'],float)  # (13,116)
peak=np.nanmax(np.abs(A),axis=0)
def net(i):
    n=ROI_NAME.get(i,'')
    if any(k in n for k in ['Somatomotor','Visual','Aud']): return 'sensory/motor'
    if any(k in n for k in ['DMN','Temporoparietal','Limbic','OFC','TempPole']): return 'social/affective'
    return 'other'
G={'sensory/motor':[],'social/affective':[],'other':[]}
for i in range(116): G[net(i)].append(peak[i])
for k,v in G.items(): print(f'  {k:16s} n={len(v):3d} mean peak|r|={np.mean(v):.3f} max={np.max(v):.3f}')
print('  sensory>social one-sided p =', round(mannwhitneyu(G['sensory/motor'],G['social/affective'],alternative='greater').pvalue,3))
print('  Top-8 aligned ROIs:')
for i in np.argsort(peak)[::-1][:8]: print(f'    {i:3d} |r|={peak[i]:.3f} {ROI_NAME.get(i,"?")}')


  sensory/motor    n= 27 mean peak|r|=0.033 max=0.068
  social/affective n= 30 mean peak|r|=0.028 max=0.058
  other            n= 59 mean peak|r|=0.022 max=0.060
  sensory>social one-sided p = 0.12
  Top-8 aligned ROIs:
     60 |r|=0.068 RH Somatomotor (S2_1)
     14 |r|=0.060 LH DorsalAttn (SPL_2)
     98 |r|=0.058 RH Temporoparietal (2)
      2 |r|=0.055 LH Visual (ExStr_3)
      9 |r|=0.053 LH Somatomotor (Aud_1)
     61 |r|=0.051 RH Somatomotor (S2_2)
     78 |r|=0.050 RH Limbic (TempPole_1)
     97 |r|=0.048 RH Temporoparietal (1)


## §3 · Variance partitioning — does the model explain neural synchrony that behavior doesn't?

Commonality analysis in IS-RSA space. For each ROI, predict **neural** pairwise similarity from **impression** similarity (USE = behavior) and **sentiment** similarity (RoBERTa = model); partition R² into unique-behavior, unique-model, shared.

**Estimator note (fixed 2026-07-18):** the first version used OLS `lstsq` on z-scored values and threw `divide-by-zero / overflow / invalid` warnings on degenerate ROIs. Replaced with a **correlation-based commonality decomposition on Spearman ranks** — closed-form, numerically stable, and consistent with the rank-based IS-RSA itself:

```
R²_full = (r_yI² + r_yS² − 2·r_yI·r_yS·r_IS) / (1 − r_IS²)
unique_beh = R²_full − r_yS²   unique_model = R²_full − r_yI²   shared = r_yI² + r_yS² − R²_full
```

> [!note] Result (2026-07-18, 0 degenerate ROIs)
> Social ROIs — uniq_beh vs uniq_MODEL: **60:** 0.0085 vs 0.0019 · **98:** 0.0035 vs 0.0003 · **24:** 0.0009 vs 0.0001 · **48:** 0.0003 vs 0.0006 · **91:** ~0 vs ~0.
> Across all 116 ROIs: uniq_BEH mean **+0.00065** (max 0.0099) vs uniq_MODEL mean **+0.00035** (max 0.0020).
>
> **Read:** behaviour carries roughly **2–5× the unique variance** the model does in the social ROIs, and the model's unique contribution never exceeds 0.002. Consistent with `04b`/`04b.1`: the model adds essentially nothing over human impressions. **Caveat:** total R² is tiny for *both* predictors, so this is 'consistent with' rather than strong positive evidence — IS-RSA effects are small and pair-level data is noisy.


In [8]:
import numpy as np, json as _json, os
from scipy.stats import rankdata
from config import JIN_REPO
from helpers import _pair_mask, rearrange_new, NROI
neural=np.load(os.path.join(JIN_REPO,'data/brain/similarity/neuralISC_byevent.npy'),allow_pickle=True).item()
USE =np.load(f'{JIN_REPO}/data/beh/similarity/impressions_byrun_bychar.npy',allow_pickle=True).item()   # behaviour
SENT=np.load('results/jinrep/03__sentiment_sim_byrun_bychar.npy',allow_pickle=True).item()              # model
overlap=_json.load(open('results/jinrep/03__isrsa_subject_order.json'))
masks={g:_pair_mask(g,overlap[str(g)]) for g in [1,2,3]}
def rk(x):
    x=np.asarray(x,float); out=np.full_like(x,np.nan); m=~np.isnan(x)
    if m.sum()>1: out[m]=rankdata(x[m])
    return out
def commonality(y,I,S):
    m=~(np.isnan(y)|np.isnan(I)|np.isnan(S))
    if m.sum()<10: return dict(uniq_beh=np.nan,uniq_model=np.nan,shared=np.nan,full=np.nan)
    y,I,S=rk(y[m]),rk(I[m]),rk(S[m])
    def r(a,b):
        sa,sb=np.std(a),np.std(b)
        return 0.0 if sa<1e-12 or sb<1e-12 else float(np.corrcoef(a,b)[0,1])
    ryI,ryS,rIS=r(y,I),r(y,S),r(I,S); den=1-rIS**2
    if den<1e-8: return dict(uniq_beh=np.nan,uniq_model=np.nan,shared=np.nan,full=np.nan)
    Rf=min(max((ryI**2+ryS**2-2*ryI*ryS*rIS)/den,0.0),1.0)
    return dict(uniq_beh=Rf-ryS**2, uniq_model=Rf-ryI**2, shared=ryI**2+ryS**2-Rf, full=Rf)
impr=[np.concatenate([np.asarray(USE[g,r+1],float)[:,masks[g]] for r in range(10)]) for g in range(1,4)]
sent=[np.concatenate([np.asarray(SENT[g,r+1],float) for r in range(10)]) for g in range(1,4)]
social=[24,48,60,91,98]; res={}
for roi in range(NROI):
    yv,iv,sv=[],[],[]
    for gi,g in enumerate([1,2,3]):
        per=[(np.vstack([np.nanmean(neural[roi+1,g,r],0)]*4) if r==6 else rearrange_new(g,r+1,neural[roi+1,g,r]))[:,masks[g]] for r in range(10)]
        yv.append(np.concatenate(per).ravel()); iv.append(impr[gi].ravel()); sv.append(sent[gi].ravel())
    res[roi]=commonality(np.concatenate(yv),np.concatenate(iv),np.concatenate(sv))
print('ROI  uniq_beh   uniq_MODEL   shared     full')
for roi in social:
    o=res[roi]; print(f"  {roi:3d}  {o['uniq_beh']:+.5f}  {o['uniq_model']:+.5f}  {o['shared']:+.5f}  {o['full']:.5f}")
um=np.array([res[r]['uniq_model'] for r in range(NROI)],float); ub=np.array([res[r]['uniq_beh'] for r in range(NROI)],float)
print(f'across 116 ROIs: uniq_MODEL mean={np.nanmean(um):+.5f} max={np.nanmax(um):+.5f} | uniq_BEH mean={np.nanmean(ub):+.5f} max={np.nanmax(ub):+.5f}')
print('degenerate ROIs:',int(np.isnan(um).sum()))
np.save('results/jinrep/10__commonality.npy',res,allow_pickle=True)


ROI  uniq_beh   uniq_MODEL   shared     full
   24  +0.00091  +0.00006  -0.00003  0.00094
   48  +0.00028  +0.00059  -0.00006  0.00080
   60  +0.00850  +0.00185  -0.00059  0.00976
   91  +0.00002  +0.00002  +0.00000  0.00004
   98  +0.00350  +0.00032  +0.00020  0.00402
across 116 ROIs: uniq_MODEL mean=+0.00035 max=+0.00198 | uniq_BEH mean=+0.00065 max=+0.00986
degenerate ROIs: 0


## §4 · Does emotion interrupt character understanding?  (confound-controlled + multiverse)

**v1 (discarded).** Split the four characters 2-vs-2 by emotional intensity — perfectly confounds emotion with **character identity**. Reported mentalizing *stronger* for high-emotion. Artifact.

**v2 (below).** Scene-level intensity from run-resolved `char_valence_composite`, median-split **within character** so each character contributes to both bins; 1000 permutations shuffling labels within character. Direction reversed to *support* the hypothesis (mean diff −0.057) but **p = .119, n.s.**

**v3 — the multiverse (§4b).** Rather than chase the v2 trend, we ran the whole specification space: emotion treated **continuously** (no arbitrary cutoff, uses all 120 scenes) × 4 intensity metrics × 3 mentalizing ROIs + pooled = **16 specifications, all reported**.

> [!warning] Verdict — the trend does not survive
> Only **4 of 16** specifications run in the hypothesised (negative) direction — a coin flip. Median rho = **+0.056** (slightly *opposite*), range [−0.095, +0.149], and **0 of 16** reach p<.05.
>
> **Read: no evidence that emotion interrupts character understanding.** The v2 median-split trend was specification-dependent — it appears under one binarisation and disappears under the better-powered continuous estimator across all four emotion metrics. Report §4 as a null.
>
> *Note:* v2 and §4b use different decompositions (v2 correlates across scenes per subject-pair; §4b computes one alignment per scene across pairs), so they aren't strictly apples-to-apples — but §4b uses all 120 scenes and every metric, and v2 was itself n.s., so the null holds either way.
>
> **Methodological keeper:** same data, same ROIs — character-split said one thing, scene-split said the opposite, multiverse said nothing. Worth a line in the methods about why the confound-controlled multiverse is the reportable analysis.


In [9]:
import numpy as np, json as _json, os, pandas as pd
from scipy.stats import rankdata
from config import JIN_REPO
from helpers import _pair_mask, rearrange_new, z2r, r2z
CHAR=['jack','kate','randall','kevin']; MENT=[91,97,98]; N_PERM,SEED=1000,42
neural=np.load(os.path.join(JIN_REPO,'data/brain/similarity/neuralISC_byevent.npy'),allow_pickle=True).item()
USE=np.load(f'{JIN_REPO}/data/beh/similarity/impressions_byrun_bychar.npy',allow_pickle=True).item()
overlap=_json.load(open('results/jinrep/03__isrsa_subject_order.json')); masks={g:_pair_mask(g,overlap[str(g)]) for g in [1,2,3]}
# scene-level emotional intensity, median-split WITHIN character (decouples emotion from character identity)
t=pd.read_csv('results/step1/01__targets.csv'); t['Character']=t.Character.str.lower().str.strip()
t['intensity']=(t['char_valence_composite']-4).abs()
hi_flag={}
for (g,ch),sub in t.groupby(['group','Character']):
    med=sub['intensity'].median()
    for _,row in sub.iterrows(): hi_flag[(int(g),ch,int(row.Run))]=bool(row['intensity']>med)
def unit_labels(g):                      # stacked order: unit index = run*4 + char
    lab=np.zeros(40,bool)
    for r in range(10):
        for c,ch in enumerate(CHAR): lab[r*4+c]=hi_flag.get((g,ch,r+1),False)
    return lab
labels={g:unit_labels(g) for g in [1,2,3]}
print('high-emotion scenes per group:',{g:int(labels[g].sum()) for g in [1,2,3]},'of 40')
beh={g:np.concatenate([np.asarray(USE[g,r+1],float)[:,masks[g]] for r in range(10)]) for g in [1,2,3]}
brain={roi:{g:np.concatenate([(np.vstack([np.nanmean(neural[roi+1,g,r],0)]*4) if r==6 else rearrange_new(g,r+1,neural[roi+1,g,r]))[:,masks[g]] for r in range(10)]) for g in [1,2,3]} for roi in MENT}
def spear_cols(A,B):
    def rk(M):
        R=np.apply_along_axis(lambda v: rankdata(v) if not np.all(np.isnan(v)) else v,0,M)
        return R-np.nanmean(R,0,keepdims=True)
    Ar,Br=rk(A),rk(B); num=np.nansum(Ar*Br,0); den=np.sqrt(np.nansum(Ar**2,0)*np.nansum(Br**2,0))
    with np.errstate(all='ignore'): return np.where(den>1e-9,num/den,np.nan)
def mean_r(roi,sel):
    rv=[spear_cols(z2r(brain[roi][g][sel[g]]), beh[g][sel[g]]) for g in [1,2,3] if sel[g].sum()>=4]
    return z2r(np.nanmean(r2z(np.concatenate(rv))))
hi={g:labels[g] for g in [1,2,3]}; lo={g:~labels[g] for g in [1,2,3]}
obs={roi:(mean_r(roi,lo),mean_r(roi,hi)) for roi in MENT}
print('\nROI   low-emotion r   high-emotion r   diff(high-low)')
for roi in MENT:
    l,h=obs[roi]; print(f'  {roi:3d}      {l:+.4f}        {h:+.4f}       {h-l:+.4f}')
obs_diff=np.mean([obs[r][1]-obs[r][0] for r in MENT]); print(f'\nmean mentalizing diff (high-low) = {obs_diff:+.4f}')
rng=np.random.default_rng(SEED); null=np.empty(N_PERM)
for it in range(N_PERM):
    perm={}
    for g in [1,2,3]:
        lab=np.zeros(40,bool)
        for c in range(4):
            idx=[r*4+c for r in range(10)]; k=int(labels[g][idx].sum())
            for p in rng.choice(10,size=k,replace=False): lab[idx[p]]=True
        perm[g]=lab
    ph={g:perm[g] for g in [1,2,3]}; pl={g:~perm[g] for g in [1,2,3]}
    null[it]=np.mean([mean_r(r,ph)-mean_r(r,pl) for r in MENT])
p=(np.sum(np.abs(null)>=abs(obs_diff))+1)/(N_PERM+1)
print(f'permutation p = {p:.3f}  (null mean={np.nanmean(null):+.4f} sd={np.nanstd(null):.4f})')
print('=> ' + ('NO significant evidence that emotion interrupts understanding (trend only)' if p>=0.05 else 'significant'))
np.save('results/step6/10__emotion_split_mentalizing.npy',{'obs':obs,'obs_diff':obs_diff,'p':p,'null':null},allow_pickle=True)


high-emotion scenes per group: {1: 20, 2: 20, 3: 20} of 40

ROI   low-emotion r   high-emotion r   diff(high-low)
   91      +0.0109        -0.0112       -0.0221
   97      +0.1048        +0.0048       -0.1000
   98      +0.0758        +0.0257       -0.0501

mean mentalizing diff (high-low) = -0.0574


/var/folders/lk/34qrvdj11cq5ln8vscwdnrg00000gn/T/ipykernel_8914/2553158609.py:28: RuntimeWarning: Mean of empty slice
  return R-np.nanmean(R,0,keepdims=True)


permutation p = 0.119  (null mean=-0.0006 sd=0.0364)
=> NO significant evidence that emotion interrupts understanding (trend only)


### §4b · Specification curve (continuous emotion, all 16 specs reported)

Per-scene IS-RSA alignment (across subject-pairs, one value per scene) vs continuous scene emotional intensity. Negative rho = emotion interrupts. Nothing dropped, nothing cherry-picked.


In [10]:
import numpy as np, json as _json, os, pandas as pd
from scipy.stats import spearmanr
from config import JIN_REPO
from helpers import _pair_mask, rearrange_new, z2r
CHAR=['jack','kate','randall','kevin']; MENT=[91,97,98]
neural=np.load(os.path.join(JIN_REPO,'data/brain/similarity/neuralISC_byevent.npy'),allow_pickle=True).item()
USE=np.load(f'{JIN_REPO}/data/beh/similarity/impressions_byrun_bychar.npy',allow_pickle=True).item()
overlap=_json.load(open('results/jinrep/03__isrsa_subject_order.json')); masks={g:_pair_mask(g,overlap[str(g)]) for g in [1,2,3]}
t=pd.read_csv('results/step1/01__targets.csv'); t['Character']=t.Character.str.lower().str.strip()
beh={g:np.concatenate([np.asarray(USE[g,r+1],float)[:,masks[g]] for r in range(10)]) for g in [1,2,3]}
def per_scene_alignment(roi):
    B={g:np.concatenate([(np.vstack([np.nanmean(neural[roi+1,g,r],0)]*4) if r==6 else rearrange_new(g,r+1,neural[roi+1,g,r]))[:,masks[g]] for r in range(10)]) for g in [1,2,3]}
    out={}
    for g in [1,2,3]:
        a=np.full(40,np.nan)
        for u in range(40):
            x,y=z2r(B[g][u]),beh[g][u]; m=~(np.isnan(x)|np.isnan(y))
            if m.sum()>4: a[u]=spearmanr(x[m],y[m])[0]
        out[g]=a
    return out
def intensity_vec(metric,center):
    out={}
    for g in [1,2,3]:
        v=np.full(40,np.nan)
        for r in range(10):
            for c,ch in enumerate(CHAR):
                s=t[(t.group==g)&(t.Character==ch)&(t.Run==r+1)]
                if len(s): v[r*4+c]=abs(float(s[metric].iloc[0])-center)
        out[g]=v
    return out
ALIGN={roi:per_scene_alignment(roi) for roi in MENT}
metrics=[('char_valence_composite',4.0),('positive_emotion',4.0),('affect_cluster',4.0),('PC1',0.0)]
spec=[]
for mname,center in metrics:
    I=intensity_vec(mname,center)
    for roi in MENT:
        a=np.concatenate([ALIGN[roi][g] for g in [1,2,3]]); iv=np.concatenate([I[g] for g in [1,2,3]])
        m=~(np.isnan(a)|np.isnan(iv)); rho,p=spearmanr(a[m],iv[m]); spec.append((mname,roi,rho,p))
        print(f'  {mname:24s} ROI {roi:3d}  rho={rho:+.3f} p={p:.3f}')
    a=np.concatenate([np.nanmean([ALIGN[r][g] for r in MENT],0) for g in [1,2,3]]); iv=np.concatenate([I[g] for g in [1,2,3]])
    m=~(np.isnan(a)|np.isnan(iv)); rho,p=spearmanr(a[m],iv[m]); spec.append((mname,'POOLED',rho,p))
    print(f'  {mname:24s} POOLED   rho={rho:+.3f} p={p:.3f}\n')
rs=np.array([s[2] for s in spec]); ps=np.array([s[3] for s in spec])
print('SPEC CURVE: %d specs | negative (hypothesis dir): %d/%d | median rho=%+.3f | range [%+.3f,%+.3f] | p<.05: %d/%d'%(
      len(spec),int((rs<0).sum()),len(rs),np.median(rs),rs.min(),rs.max(),int((ps<0.05).sum()),len(ps)))
np.save('results/step6/10__emotion_specification_curve.npy',{'spec':spec},allow_pickle=True)


  char_valence_composite   ROI  91  rho=+0.046 p=0.614
  char_valence_composite   ROI  97  rho=+0.094 p=0.305
  char_valence_composite   ROI  98  rho=+0.149 p=0.105
  char_valence_composite   POOLED   rho=+0.147 p=0.110

  positive_emotion         ROI  91  rho=-0.005 p=0.957
  positive_emotion         ROI  97  rho=-0.038 p=0.683
  positive_emotion         ROI  98  rho=+0.042 p=0.648
  positive_emotion         POOLED   rho=+0.005 p=0.955

  affect_cluster           ROI  91  rho=+0.080 p=0.383
  affect_cluster           ROI  97  rho=-0.088 p=0.337
  affect_cluster           ROI  98  rho=+0.112 p=0.222
  affect_cluster           POOLED   rho=+0.064 p=0.485

  PC1                      ROI  91  rho=+0.129 p=0.160
  PC1                      ROI  97  rho=-0.095 p=0.303
  PC1                      ROI  98  rho=+0.090 p=0.330
  PC1                      POOLED   rho=+0.048 p=0.601

SPEC CURVE: 16 specs | negative (hypothesis dir): 4/16 | median rho=+0.056 | range [-0.095,+0.149] | p<.05: 0/16
